# Beirut CLIP embeddings — GPU on Colab

Replaces Step 2 of `beirut_analysis.ipynb` with a ~20-minute GPU run on Colab.
Downloads images **directly from Mapillary URLs** on Colab's fast network — no
need to upload 13 GB of JPEGs from your laptop.

## How to use

1. Open in [colab.research.google.com](https://colab.research.google.com) (File → Upload notebook)
2. **Runtime → Change runtime type → T4 GPU** (free)
3. Run all cells. In Cell 2, you can upload multiple files at once:
   - `beirut_metadata.csv.gz` (required — has the URLs)
   - `embeddings.npy` + `embeddings_ids.txt` (optional — to resume a previous run)
4. When the last cell finishes, you'll get a download dialog with `embeddings.npy` and `embeddings_ids.txt`
5. Save them into your project folder next to `beirut_analysis.ipynb`

**Note:** Mapillary signs `thumb_2048_url` for ~24-48h. If you generated the CSV more than a day ago, re-run Step 0 of `beirut_analysis.ipynb` locally first, then upload the fresh CSV here.

In [ ]:
# Cell 1 — install + GPU check
!pip install -q open_clip_torch
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}   torch: {torch.__version__}')

In [ ]:
# Cell 2 — upload files
#
# Drag in ALL of these at once (in the same upload dialog):
#   beirut_metadata.csv.gz                     (REQUIRED — has the image URLs)
#   embeddings.npy + embeddings_ids.txt        (OPTIONAL — to resume a partial run)
#
# This cell auto-detects what you uploaded; you can also re-run it later if you
# only need to add one of the files.
from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np

uploaded = files.upload()
for name, data in uploaded.items():
    print(f'  uploaded: {name}  ({len(data)/1e6:.1f} MB)')

# Find the metadata CSV — either freshly uploaded or already on disk from earlier
csv_candidates = list(Path('.').glob('*.csv.gz')) + list(Path('.').glob('*.csv'))
csv_candidates = [p for p in csv_candidates if 'metadata' in p.name.lower() or 'beirut' in p.name.lower()]
if not csv_candidates:
    csv_candidates = list(Path('.').glob('*.csv.gz')) + list(Path('.').glob('*.csv'))
assert csv_candidates, 'No CSV file found. Please upload beirut_metadata.csv.gz and re-run this cell.'
csv_path = csv_candidates[0]
print(f'\nmetadata file: {csv_path}')

df = pd.read_csv(csv_path)
df['id'] = df['id'].astype(str)
df = df[df['thumb_2048_url'].notna()].reset_index(drop=True)
print(f'metadata: {len(df):,} rows with URLs')

# Detect resume baseline
if Path('embeddings.npy').exists() and Path('embeddings_ids.txt').exists():
    prior = np.load('embeddings.npy')
    with open('embeddings_ids.txt') as f:
        prior_ids = [l.strip() for l in f if l.strip()]
    if len(prior) == len(prior_ids):
        print(f'\nresume baseline detected: {len(prior_ids):,} embeddings already done')
        print(f'Cell 3 will only embed the missing {max(0, len(df) - len(prior_ids)):,} images')
    else:
        print(f'\nWARN: embeddings.npy / embeddings_ids.txt shape mismatch ({len(prior)} vs {len(prior_ids)}) — will start fresh')
else:
    print('\n(no resume files — Cell 3 will start fresh)')

In [ ]:
# Cell 3 — stream embed: parallel download → batch GPU encode → save incrementally
import os, time, io
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import requests
from PIL import Image
import open_clip

EMB_PATH = Path('embeddings.npy')
IDS_PATH = Path('embeddings_ids.txt')

if EMB_PATH.exists() and IDS_PATH.exists():
    emb_all = np.load(EMB_PATH)
    with IDS_PATH.open() as f:
        emb_ids = [l.strip() for l in f if l.strip()]
    if len(emb_all) != len(emb_ids):
        print(f'shape mismatch ({len(emb_all)} vs {len(emb_ids)}) — starting fresh')
        emb_all = np.zeros((0, 512), dtype=np.float32)
        emb_ids = []
    else:
        print(f'resuming from {len(emb_ids):,} existing embeddings')
else:
    emb_all = np.zeros((0, 512), dtype=np.float32)
    emb_ids = []

done = set(emb_ids)
df_todo = df[~df['id'].isin(done)].reset_index(drop=True)
print(f'to embed: {len(df_todo):,}')

if len(df_todo) == 0:
    print('All done already — skip to Cell 4 to download the files.')
else:
    print('Loading CLIP ViT-B-32 on GPU...')
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
    model.eval().cuda()

    def fetch_and_preprocess(row):
        try:
            r = requests.get(row['thumb_2048_url'], timeout=30)
            r.raise_for_status()
            img = Image.open(io.BytesIO(r.content)).convert('RGB')
            return row['id'], preprocess(img)
        except Exception:
            return row['id'], None

    CHUNK = 256
    GPU_BATCH = 64
    DL_THREADS = 32

    t0 = time.time()
    n_fail = 0
    for chunk_start in range(0, len(df_todo), CHUNK):
        rows = df_todo.iloc[chunk_start:chunk_start + CHUNK].to_dict('records')
        with ThreadPoolExecutor(max_workers=DL_THREADS) as ex:
            results = list(ex.map(fetch_and_preprocess, rows))
        valid = [(iid, t) for iid, t in results if t is not None]
        n_fail += len(results) - len(valid)
        if not valid: continue

        for bs in range(0, len(valid), GPU_BATCH):
            batch = valid[bs:bs + GPU_BATCH]
            ids = [b[0] for b in batch]
            tens = torch.stack([b[1] for b in batch]).cuda(non_blocking=True)
            with torch.no_grad():
                feat = model.encode_image(tens)
                feat = feat / feat.norm(dim=-1, keepdim=True)
            emb_all = np.vstack([emb_all, feat.cpu().numpy().astype(np.float32)])
            emb_ids.extend(ids)

        np.save(EMB_PATH, emb_all)
        with IDS_PATH.open('w') as f:
            for i in emb_ids: f.write(i + '\n')

        elapsed = time.time() - t0
        processed = chunk_start + len(rows)
        rate = processed / max(1e-6, elapsed)
        eta = (len(df_todo) - processed) / max(1e-6, rate) / 60
        print(f'  {processed:,}/{len(df_todo):,}   rate={rate:.1f} img/s   ETA={eta:.1f} min   fail={n_fail}')

    print(f'\nDONE.  embeddings: {emb_all.shape}   ids: {len(emb_ids):,}   failed: {n_fail}   took {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 4 — download the two files back to your laptop
from google.colab import files
files.download('embeddings.npy')
files.download('embeddings_ids.txt')